In [1]:
import os
import optuna
import pandas as pd
from optuna.trial import TrialState
from optuna.study import StudyDirection

In [2]:
import os
import pandas as pd
import optuna
from optuna.trial import TrialState

def best_trial(
    db_path: str,
    study_name: str,
    n_trials: int = 50,
    direction: str = 'min'
):
    study = optuna.load_study(
        study_name=study_name,
        storage=f"sqlite:///{db_path}"
    )

    first_n = sorted(study.trials, key=lambda t: t.number)[:n_trials]
    completed = [t for t in first_n if t.state == TrialState.COMPLETE]

    if not completed:
        raise ValueError("No completed trials found in the first N trials.")

    if direction == 'min':
        best = min(completed, key=lambda t: t.value)
    elif direction == 'max':
        best = max(completed, key=lambda t: t.value)
    else:
        raise ValueError("Direction must be 'min' or 'max'")

    return {
        'trial_number': best.number,
        'value': best.value,
        'learning_rate': best.params.get('learning_rate'),
        'weight_decay': best.params.get('weight_decay'),
        'pooling': best.params.get('pooling'),
        'use_numeric': best.params.get('use_numeric')
    }

# Loop and collect results
dbs_path = '/scratch/sas10092/ehr-foundation/models/optuna_dbs/'
dbs = os.listdir(dbs_path)

rows = []
for db in dbs:
    arch = db.split('_')[0]
    if arch == 'big':
        arch = 'big_bird'
    task = db[len(arch)+1:-3]

    result = best_trial(
        db_path=os.path.join(dbs_path, db),
        study_name=arch,
        n_trials=25,
        direction='min'
    )
    row = {
        'arch': arch,
        'task': task,
        **result
    }
    rows.append(row)

# Convert to DataFrame and reorder columns
df = pd.DataFrame(rows)
df = df[['arch', 'task', 'trial_number', 'value', 'learning_rate', 'weight_decay', 'pooling', 'use_numeric']]

# Optionally save to CSV
# df.to_csv("best_trials_summary.csv", index=False)

# Show result
df = df.sort_values(['task'])
df = df[df.arch != 'bert']
df = df.reset_index(drop=True)

In [3]:
df

,arch,task,trial_number,value,learning_rate,weight_decay,pooling,use_numeric
0,remed,bert-ft_y_icu_readmit_30,5,0.159270,0.000683,NaN,None,None
1,genhpf,bert-ft_y_icu_readmit_30,16,0.149513,0.000085,NaN,None,None
2,descemb,bert-ft_y_icu_readmit_30,9,0.154141,0.000280,NaN,None,None
3,genhpf,bert-ft_y_los_7,10,0.292799,0.000016,NaN,None,None
4,descemb,bert-ft_y_los_7,19,0.291670,0.000073,NaN,None,None
5,remed,bert-ft_y_mort,7,0.220950,0.000221,NaN,None,None
6,descemb,bert-ft_y_mort,23,0.167951,0.000013,NaN,None,None
7,genhpf,bert-ft_y_mort,12,0.168080,0.000099,NaN,None,None
8,descemb,bert-ft_y_mort_1yr,15,0.256196,0.000326,NaN,None,None
9,remed,bert-ft_y_mort_1yr,22,0.277947,0.000157,NaN,None,None


In [55]:
# change this
study = df.iloc[1]
study

arch             big_bird
task              y_los_7
trial_number           25
value            0.280409
learning_rate    0.000049
weight_decay     0.001957
pooling               cls
use_numeric         False
Name: 5, dtype: object

In [56]:
import os